# Music Annotation Project: Notebook 68
## Pilot Component-to-Semantic-Region Alignment Audit with Human Review Capture

**Runtime:** GPU recommended.

This notebook tests whether the frozen 59R6 semantic bundle performs better when Pilot 09 components are represented as coherent regions. It compares `RAW`, `59R5_MERGED`, and `ALIGNMENT_PROPOSAL` representations using unchanged models, calibrators, classes, and thresholds.

It also includes a downstream human reviewer. Human labels, grouping judgements, and correction actions are persisted separately and never alter inference in this run. No automatic correction or production promotion is authorised.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy torch torchvision gradio pillow matplotlib psutil

In [ ]:
import os, re, json, hashlib, random, shutil, sys, platform, gc, zipfile
from pathlib import Path
from datetime import datetime, timezone
import cv2, numpy as np, pandas as pd, torch
from torch import nn
from PIL import Image
os.environ['PYTHONHASHSEED']='42'
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
PILOT_ROOT=ROOT/'production_pilot'/'runs'; R6_ROOT=ROOT/'region_semantic_59r6'; OUTPUT_ROOT=ROOT/'notebook68_component_semantic_alignment_audit'
PILOT09_RUN_OVERRIDE=None; R6_RUN_OVERRIDE=None
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'; SEED=42
SCALES=(33,97,257); CHANNELS=['observed','normalised','region_mask','region_distance','edge','valid_image']; PAD=131
MERGE_GAP=8; MERGE_DILATION=3
ALIGN_H_GAP=55; ALIGN_V_GAP=30; ALIGN_OVERLAP=.20; ALIGN_MAX_MEMBERS=12; ALIGN_MAX_BBOX_PAGE=.20; ALIGN_DARK=210; ALIGN_DENSITY=.006
PAGE_ALLOWLIST=None; MAX_PAGES=None; ENABLE_HUMAN_REVIEW=True; PUBLIC_LINK=True; MAX_REVIEW=300
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook68'; OUT=OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; AUDIT=OUT/'audit'; CROPS=OUT/'review_crops'; HUMAN=OUT/'human_review'; UPLOAD=OUT/'upload'
for d in [OUT,INPUTS,TABLES,AUDIT,CROPS,HUMAN,UPLOAD]: d.mkdir(parents=True,exist_ok=False)
print('Device:',DEVICE,'Output:',OUT)

In [ ]:
def require(x,msg):
    if not x: raise RuntimeError(msg)
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def sha256_file(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''): h.update(b)
    return h.hexdigest()
def array_hash(a):
    a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def atomic_json(p,x):
    p.write_text(json.dumps(x,indent=2,sort_keys=True,default=str))
def canon(x): return str(x).strip().upper().replace('-','_').replace(' ','_')
PILOT=Path(PILOT09_RUN_OVERRIDE) if PILOT09_RUN_OVERRIDE else latest_complete(PILOT_ROOT,['tables/component_evidence.csv','pilot_manifest.json','audit'])
R6=Path(R6_RUN_OVERRIDE) if R6_RUN_OVERRIDE else latest_complete(R6_ROOT,['frozen_bundle/freeze_manifest.json','frozen_bundle/policy.json','frozen_bundle/channel_schema.json','frozen_bundle/replay_test.json'])
require(PILOT is not None,'Complete Pilot 09 run not found'); require(R6 is not None,'Complete 59R6 run not found')
print('Pilot:',PILOT); print('59R6:',R6)

## 1. Frozen bundle, exact architecture, and strict checkpoint load

In [ ]:
FREEZE=R6/'frozen_bundle'; freeze=json.loads((FREEZE/'freeze_manifest.json').read_text()); policy=json.loads((FREEZE/'policy.json').read_text()); schema=json.loads((FREEZE/'channel_schema.json').read_text()); replay=json.loads((FREEZE/'replay_test.json').read_text())
require(policy.get('policy_id')=='59R6-shadow-v1','Unexpected policy'); require(replay.get('passed') is True,'Replay failed'); require(schema.get('channels')==CHANNELS,'Channel mismatch'); require(tuple(schema.get('scales',[]))==SCALES,'Scale mismatch')
model_paths=[]
for item in freeze.get('models',[]):
    raw=item.get('frozen_path') or item.get('path') or item.get('checkpoint')
    if raw:
        p=Path(raw); p=p if p.is_absolute() else FREEZE/p
        if p.exists(): model_paths.append(p)
if not model_paths: model_paths=sorted(FREEZE.rglob('*fold*.pt'))
require(len(model_paths)==5,f'Expected 5 models, found {len(model_paths)}')
class Branch(nn.Module):
    def __init__(self,number_of_channels=6):
        super().__init__(); self.network=nn.Sequential(nn.Conv2d(number_of_channels,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1))
    def forward(self,x): return self.network(x).flatten(1)
class MultiCNN(nn.Module):
    def __init__(self,nbranches,nclasses):
        super().__init__(); self.branches=nn.ModuleList([Branch() for _ in range(nbranches)]); self.head=nn.Sequential(nn.Linear(64*nbranches,96),nn.ReLU(),nn.Dropout(.15),nn.Linear(96,nclasses))
    def forward(self,xs): return self.head(torch.cat([b(x) for b,x in zip(self.branches,xs)],1))
ensemble=[]
for p in model_paths:
    payload=torch.load(p,map_location='cpu'); state=payload.get('state_dict',payload); classes=payload.get('classes') or policy.get('classes'); require(classes is not None,f'{p.name}: classes missing'); classes=[canon(x) for x in classes]
    model=MultiCNN(3,len(classes)); missing=sorted(set(model.state_dict())-set(state)); unexpected=sorted(set(state)-set(model.state_dict())); require(not missing and not unexpected,f'{p.name}: missing={missing[:8]} unexpected={unexpected[:8]}'); model.load_state_dict(state,strict=True); model.to(DEVICE).eval()
    m=re.search(r'fold(\d+)',p.name); require(m is not None,f'Fold missing in {p.name}'); fold=int(m.group(1)); cp=FREEZE/f'temperature_fold{fold}.json'; require(cp.exists(),f'Calibrator missing {fold}'); temp=float(json.loads(cp.read_text())['temperature'])
    ensemble.append({'fold':fold,'model':model,'classes':classes,'temperature':temp,'checkpoint':p})
rosters={tuple(x['classes']) for x in ensemble}; require(len(rosters)==1,'Class roster mismatch'); CLASSES=list(next(iter(rosters))); ensemble=sorted(ensemble,key=lambda x:x['fold'])
atomic_json(AUDIT/'59R6_bundle_resolution.json',{'bundle_id':freeze.get('bundle_id'),'policy':policy,'schema':schema,'replay':replay,'models':[{'path':str(p),'sha256':sha256_file(p)} for p in model_paths]})
print(CLASSES,[(x['fold'],x['temperature']) for x in ensemble])

## 2. Resolve Pilot 09 components and authoritative observed pages

In [ ]:
components=pd.read_csv(PILOT/'tables'/'component_evidence.csv',low_memory=False); need={'page','component','x','y','bbox_w','bbox_h','passes_frozen_structural_threshold','pilot_category'}; require(not (need-set(components)),f'Missing columns {sorted(need-set(components))}')
components['page']=components.page.astype(str); components['component']=pd.to_numeric(components.component).astype(int)
for c in ['x','y','bbox_w','bbox_h']: components[c]=pd.to_numeric(components[c]).astype(int)
require(not components.duplicated(['page','component']).any(),'Duplicate component IDs')
page_rows=[]
for page in sorted(components.page.unique()):
    hits=list((PILOT/'audit').glob(f'{page}_input_resolution.json')); require(len(hits)==1,f'{page}: input resolution record count {len(hits)}'); rec=json.loads(hits[0].read_text()); op=Path(rec['observed_path']); require(op.exists(),f'{page}: observed missing'); obs=cv2.imread(str(op),0); require(obs is not None,f'{page}: unreadable'); q=components[components.page==page]; require(((q.x>=0)&(q.y>=0)&(q.x+q.bbox_w<=obs.shape[1])&(q.y+q.bbox_h<=obs.shape[0])).all(),f'{page}: invalid geometry'); page_rows.append({'page':page,'observed_path':str(op),'observed_sha256':sha256_file(op),'height':obs.shape[0],'width':obs.shape[1],'components':len(q)})
page_audit=pd.DataFrame(page_rows)
if PAGE_ALLOWLIST is not None: page_audit=page_audit[page_audit.page.isin(PAGE_ALLOWLIST)]
if MAX_PAGES is not None: page_audit=page_audit.head(MAX_PAGES)
pages=page_audit.page.tolist(); components=components[components.page.isin(pages)].sort_values(['page','component']).reset_index(drop=True); page_audit.to_csv(AUDIT/'pilot09_page_input_audit.csv',index=False); shutil.copy2(PILOT/'tables'/'component_evidence.csv',INPUTS/'pilot09_component_evidence.csv')
print(len(pages),len(components))

## 3. Build RAW, frozen-merge, and alignment-proposal representations

In [ ]:
def box(r): return int(r.x),int(r.y),int(r.x+r.bbox_w),int(r.y+r.bbox_h)
def overlap(a0,a1,b0,b1): return max(0,min(a1,b1)-max(a0,b0))/max(1,min(a1-a0,b1-b0))
def gaps(a,b):
    ax0,ay0,ax1,ay1=a; bx0,by0,bx1,by1=b; return max(0,max(ax0,bx0)-min(ax1,bx1)),max(0,max(ay0,by0)-min(ay1,by1))
def frozen_link(a,b):
    hg,vg=gaps(box(a),box(b)); return hg<=MERGE_GAP and vg<=MERGE_GAP
def align_link(obs,a,b):
    aa,bb=box(a),box(b); hg,vg=gaps(aa,bb); hov=overlap(aa[0],aa[2],bb[0],bb[2]); vov=overlap(aa[1],aa[3],bb[1],bb[3]); linked=(hg<=ALIGN_H_GAP and vov>=ALIGN_OVERLAP and vg<=ALIGN_V_GAP) or (vg<=ALIGN_V_GAP and hov>=ALIGN_OVERLAP and hg<=ALIGN_H_GAP); x0=max(0,min(aa[0],bb[0])-2); y0=max(0,min(aa[1],bb[1])-2); x1=min(obs.shape[1],max(aa[2],bb[2])+2); y1=min(obs.shape[0],max(aa[3],bb[3])+2); density=float((obs[y0:y1,x0:x1]<ALIGN_DARK).mean()); return linked and density>=ALIGN_DENSITY
def group(frame,start,fn,obs=None):
    chosen={start}; frontier=[start]
    while frontier and len(chosen)<ALIGN_MAX_MEMBERS:
        i=frontier.pop(0)
        for j in frame.index:
            if j in chosen: continue
            ok=fn(frame.loc[i],frame.loc[j]) if obs is None else fn(obs,frame.loc[i],frame.loc[j])
            if ok: chosen.add(j); frontier.append(j)
            if len(chosen)>=ALIGN_MAX_MEMBERS: break
    return sorted(chosen)
def mask_for(frame,ids,shape,dilate=0):
    m=np.zeros(shape,np.uint8)
    for i in ids:
        x0,y0,x1,y1=box(frame.loc[i]); m[y0:y1,x0:x1]=1
    if dilate: m=cv2.dilate(m,np.ones((dilate,dilate),np.uint8))
    return (m>0).astype(np.uint8)
def bbox_fraction(m):
    ys,xs=np.where(m); return 0 if not len(xs) else ((xs.max()-xs.min()+1)*(ys.max()-ys.min()+1))/m.size
PAGE={}; rows=[]
for page in pages:
    op=page_audit.loc[page_audit.page==page,'observed_path'].iloc[0]; obs=cv2.imread(op,0); frame=components[components.page==page].copy(); frame.index=np.arange(len(frame)); PAGE[page]={'obs':obs,'frame':frame,'masks':{}}
    for i,r in frame.iterrows():
        groups={'RAW':[i],'59R5_MERGED':group(frame,i,frozen_link),'ALIGNMENT_PROPOSAL':group(frame,i,align_link,obs)}
        masks={'RAW':mask_for(frame,groups['RAW'],obs.shape),'59R5_MERGED':mask_for(frame,groups['59R5_MERGED'],obs.shape,MERGE_DILATION),'ALIGNMENT_PROPOSAL':mask_for(frame,groups['ALIGNMENT_PROPOSAL'],obs.shape,MERGE_DILATION)}
        if bbox_fraction(masks['ALIGNMENT_PROPOSAL'])>ALIGN_MAX_BBOX_PAGE: groups['ALIGNMENT_PROPOSAL']=groups['59R5_MERGED']; masks['ALIGNMENT_PROPOSAL']=masks['59R5_MERGED']
        cid=int(r.component); candidate=f'{page}:C{cid}'
        for rep in groups:
            m=masks[rep]; ys,xs=np.where(m); PAGE[page]['masks'][(cid,rep)]=m; rows.append({'candidate_id':candidate,'page':page,'component':cid,'representation':rep,'member_components':'|'.join(str(int(frame.loc[j].component)) for j in groups[rep]),'member_count':len(groups[rep]),'mask_pixels':int(m.sum()),'bbox_x0':int(xs.min()),'bbox_y0':int(ys.min()),'bbox_x1':int(xs.max()+1),'bbox_y1':int(ys.max()+1),'mask_sha256':array_hash(m)})
representations=pd.DataFrame(rows); representations.to_csv(TABLES/'representation_manifest.csv',index=False); print(representations.groupby('representation').member_count.describe())

## 4. Frozen tensor construction and ensemble inference

In [ ]:
def static_arrays(obs):
    bg=cv2.GaussianBlur(obs,(0,0),45); norm=cv2.divide(obs,np.maximum(bg,1),scale=255); return {'observed':obs.astype(np.float32)/255,'normalised':norm.astype(np.float32)/255,'edge':cv2.Canny(obs,70,170).astype(np.float32)/255,'valid_image':np.ones(obs.shape,np.float32)}
def padded(a,mode): return np.pad(a,PAD,mode=mode)
def centre(r): return int(round(r.y+r.bbox_h/2)),int(round(r.x+r.bbox_w/2))
def crop(a,cy,cx,size):
    h=size//2; q=a[cy+PAD-h:cy+PAD+h+1,cx+PAD-h:cx+PAD+h+1]; require(q.shape==(size,size),f'Crop {q.shape}'); return q
def build_tensors(obs,mask,row):
    s=static_arrays(obs); sp={k:padded(v,'reflect' if k!='valid_image' else 'constant') for k,v in s.items()}; dist=cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5).astype(np.float32); dist=dist/max(float(dist.max()),1.0); mp=padded(mask.astype(np.float32),'constant'); dp=padded(dist,'constant'); cy,cx=centre(row); result=[]
    for size in SCALES: result.append(np.stack([crop(sp['observed'],cy,cx,size),crop(sp['normalised'],cy,cx,size),crop(mp,cy,cx,size),crop(dp,cy,cx,size),crop(sp['edge'],cy,cx,size),crop(sp['valid_image'],cy,cx,size)]).astype(np.float32))
    return result
def softmax(x): x=x-x.max(1,keepdims=True); e=np.exp(x); return e/e.sum(1,keepdims=True)
def infer(ts):
    members=[]
    for e in ensemble:
        with torch.no_grad(): logits=e['model']([torch.from_numpy(t[None]).to(DEVICE) for t in ts]).cpu().numpy()/e['temperature']
        members.append(softmax(logits)[0])
    members=np.asarray(members); mean=members.mean(0); order=np.argsort(mean)[::-1]; top_i=int(order[0]); top=float(mean[order[0]]); margin=float(mean[order[0]]-mean[order[1]]); disagreement=float(1-np.mean(np.argmax(members,1)==top_i)); decision=CLASSES[top_i]; reason='ACCEPTED'
    if top<float(policy['top_probability_threshold']): decision='UNCERTAIN'; reason='TOP_PROBABILITY_BELOW_THRESHOLD'
    elif margin<float(policy['top_two_margin_threshold']): decision='UNCERTAIN'; reason='TOP_TWO_MARGIN_BELOW_THRESHOLD'
    elif disagreement>float(policy['max_ensemble_disagreement']): decision='UNCERTAIN'; reason='ENSEMBLE_DISAGREEMENT'
    return mean,members,decision,reason,top,margin,disagreement
scores=[]; tensor_rows=[]
for n,rec in representations.iterrows():
    frame=PAGE[rec.page]['frame']; row=frame[frame.component==int(rec.component)].iloc[0]; ts=build_tensors(PAGE[rec.page]['obs'],PAGE[rec.page]['masks'][(int(rec.component),rec.representation)],row)
    for size,t in zip(SCALES,ts): tensor_rows.append({'page':rec.page,'component':int(rec.component),'representation':rec.representation,'scale':size,'tensor_sha256':array_hash(t)})
    mean,members,decision,reason,top,margin,disagreement=infer(ts); z=rec.to_dict(); z.update({'semantic_decision':decision,'semantic_reason':reason,'top_probability':top,'top_two_margin':margin,'ensemble_disagreement':disagreement,'policy_id':policy['policy_id'],'bundle_id':freeze.get('bundle_id')})
    for i,c in enumerate(CLASSES): z[f'p_{c}']=float(mean[i])
    scores.append(z)
    if (n+1)%200==0: print(n+1,'/',len(representations))
scores=pd.DataFrame(scores); scores.to_csv(TABLES/'representation_semantic_scores.csv',index=False); pd.DataFrame(tensor_rows).to_csv(AUDIT/'tensor_identity.csv',index=False); print(scores.groupby(['representation','semantic_decision']).size().unstack(fill_value=0))

## 5. Matched comparison and two-axis shadow audit

In [ ]:
sets={r:scores[scores.representation==r].set_index('candidate_id') for r in ['RAW','59R5_MERGED','ALIGNMENT_PROPOSAL']}; rows=[]
for cid in sets['RAW'].index:
    r=sets['RAW'].loc[cid]; m=sets['59R5_MERGED'].loc[cid]; a=sets['ALIGNMENT_PROPOSAL'].loc[cid]; rows.append({'candidate_id':cid,'page':r.page,'component':int(r.component),'raw_decision':r.semantic_decision,'merged_decision':m.semantic_decision,'alignment_decision':a.semantic_decision,'raw_top_probability':r.top_probability,'merged_top_probability':m.top_probability,'alignment_top_probability':a.top_probability,'raw_margin':r.top_two_margin,'merged_margin':m.top_two_margin,'alignment_margin':a.top_two_margin,'raw_disagreement':r.ensemble_disagreement,'merged_disagreement':m.ensemble_disagreement,'alignment_disagreement':a.ensemble_disagreement,'raw_to_merged_resolved':r.semantic_decision=='UNCERTAIN' and m.semantic_decision!='UNCERTAIN','raw_to_alignment_resolved':r.semantic_decision=='UNCERTAIN' and a.semantic_decision!='UNCERTAIN','alignment_regression':m.semantic_decision!='UNCERTAIN' and a.semantic_decision=='UNCERTAIN','representation_disagreement':len({r.semantic_decision,m.semantic_decision,a.semantic_decision})>1})
comparison=pd.DataFrame(rows); comparison.to_csv(TABLES/'matched_representation_comparison.csv',index=False)
summary=scores.assign(accepted=scores.semantic_decision!='UNCERTAIN').groupby('representation',as_index=False).agg(components=('candidate_id','size'),accepted=('accepted','sum'),coverage=('accepted','mean'),annotation=('semantic_decision',lambda x:int((x=='ANNOTATION').sum())),print=('semantic_decision',lambda x:int((x=='PRINT').sum())),mixed=('semantic_decision',lambda x:int((x=='MIXED').sum())),page_scan_edge=('semantic_decision',lambda x:int((x=='PAGE_SCAN_EDGE').sum())),uncertain=('semantic_decision',lambda x:int((x=='UNCERTAIN').sum())),mean_probability=('top_probability','mean'),mean_margin=('top_two_margin','mean'),mean_disagreement=('ensemble_disagreement','mean')); summary.to_csv(TABLES/'representation_summary.csv',index=False)
structural=components[['page','component','passes_frozen_structural_threshold','pilot_category']]; joined=scores.merge(structural,on=['page','component'],validate='many_to_one'); joined['structural_safe']=joined.passes_frozen_structural_threshold.astype(str).str.lower().isin(['true','1','yes'])
def route(r):
    if r.semantic_decision=='ANNOTATION' and r.structural_safe:return 'SHADOW_HIGH_HIGH_CORRECTION_CANDIDATE'
    if r.semantic_decision=='ANNOTATION':return 'SHADOW_ANNOTATION_BUT_STRUCTURALLY_RISKY'
    if r.semantic_decision=='PRINT':return 'SHADOW_LEAVE_UNCHANGED_PRINT'
    if r.semantic_decision=='MIXED':return 'SHADOW_REVIEW_MIXED'
    if r.semantic_decision=='PAGE_SCAN_EDGE':return 'SHADOW_REVIEW_PAGE_SCAN_EDGE'
    return 'SHADOW_REVIEW_SEMANTIC_UNCERTAIN'
joined['shadow_route']=joined.apply(route,axis=1); joined.to_csv(TABLES/'two_axis_representation_shadow.csv',index=False); pd.crosstab([joined.representation,joined.semantic_decision],joined.structural_safe,margins=True).to_csv(TABLES/'two_axis_matrix.csv'); print(summary.to_string(index=False))

## 6. Focused review crops and human evidence queue

In [ ]:
def render(cid):
    g=scores[scores.candidate_id==cid].set_index('representation'); page=g.iloc[0].page; comp=int(g.iloc[0].component); obs=PAGE[page]['obs']; panels=[]
    for rep,color in [('RAW',(0,150,255)),('59R5_MERGED',(255,140,0)),('ALIGNMENT_PROPOSAL',(190,0,190))]:
        r=g.loc[rep]; m=PAGE[page]['masks'][(comp,rep)]; ys,xs=np.where(m); margin=90; x0=max(0,xs.min()-margin); y0=max(0,ys.min()-margin); x1=min(obs.shape[1],xs.max()+margin+1); y1=min(obs.shape[0],ys.max()+margin+1); im=cv2.cvtColor(obs,cv2.COLOR_GRAY2RGB); cv2.rectangle(im,(int(xs.min()),int(ys.min())),(int(xs.max()),int(ys.max())),color,3); q=im[y0:y1,x0:x1]; banner=np.full((34,q.shape[1],3),255,np.uint8); cv2.putText(banner,f'{rep} {r.semantic_decision} p={r.top_probability:.3f} m={r.top_two_margin:.3f}',(4,22),cv2.FONT_HERSHEY_SIMPLEX,.42,(20,20,20),1,cv2.LINE_AA); panels.append(np.vstack([banner,q]))
    H=max(x.shape[0] for x in panels); return np.hstack([np.pad(x,((0,H-x.shape[0]),(0,0),(0,0)),constant_values=255) for x in panels])
priority=comparison.assign(priority=lambda x:np.select([x.raw_to_alignment_resolved,x.raw_to_merged_resolved,x.alignment_regression,x.representation_disagreement],[1,2,3,4],default=5)).sort_values(['priority','page','component']).head(MAX_REVIEW); queue=[]
for rank,r in enumerate(priority.itertuples(),1):
    p=CROPS/f'{rank:04d}_{r.page}_C{r.component}.png'; cv2.imwrite(str(p),cv2.cvtColor(render(r.candidate_id),cv2.COLOR_RGB2BGR)); queue.append({'rank':rank,'candidate_id':r.candidate_id,'page':r.page,'component':r.component,'priority':r.priority,'crop_path':str(p)})
review_queue=pd.DataFrame(queue).merge(comparison,on=['candidate_id','page','component']); review_queue.to_csv(TABLES/'human_review_queue.csv',index=False); print('Queue:',len(review_queue))

## 7. Human reviewer

The reviewer records a human semantic label, the most coherent representation, and the Pilot correction action. The current run remains immutable: these decisions do not change masks, probabilities, semantic decisions, structural decisions, or corrected pages.

In [ ]:
if ENABLE_HUMAN_REVIEW:
    import gradio as gr
    REVIEW_PATH=HUMAN/'human_alignment_reviews.csv'; EVENTS=HUMAN/'human_alignment_review_events.jsonl'
    def load_reviews(): return pd.read_csv(REVIEW_PATH,dtype=str).fillna('') if REVIEW_PATH.exists() else pd.DataFrame()
    def getrow(rank):
        q=review_queue[review_queue['rank']==int(rank)]; return None if q.empty else q.iloc[0]
    def show(rank):
        r=getrow(rank)
        if r is None:return None,'No row','UNCERTAIN','NONE_COHERENT','DEFER',''
        old=load_reviews(); hit=old[old.candidate_id==r.candidate_id] if len(old) else pd.DataFrame(); d=hit.iloc[-1] if len(hit) else None
        return Image.open(r.crop_path),f'{r.candidate_id} | RAW={r.raw_decision} | MERGED={r.merged_decision} | ALIGNMENT={r.alignment_decision}',d.human_semantic_label if d is not None else 'UNCERTAIN',d.grouping_judgement if d is not None else 'NONE_COHERENT',d.correction_action if d is not None else 'DEFER',d.notes if d is not None else ''
    def save(rank,semantic,grouping,action,notes):
        r=getrow(rank); require(r is not None,'Invalid row'); rec={'candidate_id':r.candidate_id,'page':r.page,'component':int(r.component),'rank':int(rank),'human_semantic_label':semantic,'grouping_judgement':grouping,'correction_action':action,'notes':notes,'reviewed_utc':datetime.now(timezone.utc).isoformat(),'raw_decision':r.raw_decision,'merged_decision':r.merged_decision,'alignment_decision':r.alignment_decision,'bundle_id':freeze.get('bundle_id'),'policy_id':policy.get('policy_id')}; old=load_reviews(); old=old[old.candidate_id!=r.candidate_id] if len(old) else old; new=pd.concat([old,pd.DataFrame([rec])],ignore_index=True); new.to_csv(REVIEW_PATH,index=False); open(EVENTS,'a').write(json.dumps(rec)+'\n'); return f'Saved {r.candidate_id}; {len(new)}/{len(review_queue)} reviewed.'
    with gr.Blocks(title='Notebook 68 reviewer') as app:
        gr.Markdown('# Notebook 68 alignment reviewer'); rank=gr.Slider(1,max(1,len(review_queue)),value=1,step=1,label='Queue row'); image=gr.Image(height=560); info=gr.Textbox(interactive=False); semantic=gr.Radio(['ANNOTATION','PRINT','MIXED','PAGE_SCAN_EDGE','UNCERTAIN'],value='UNCERTAIN',label='Human semantic label'); grouping=gr.Radio(['RAW_BEST','MERGED_BEST','ALIGNMENT_BEST','NONE_COHERENT'],value='NONE_COHERENT',label='Best grouping'); action=gr.Radio(['ACCEPT_CORRECTION','LEAVE_UNCHANGED','DEFER','EDIT_MASK_REQUIRED'],value='DEFER',label='Pilot action'); notes=gr.Textbox(label='Notes'); button=gr.Button('Save human evidence',variant='primary'); status=gr.Textbox(interactive=False); rank.change(show,rank,[image,info,semantic,grouping,action,notes]); app.load(show,rank,[image,info,semantic,grouping,action,notes]); button.click(save,[rank,semantic,grouping,action,notes],status)
    app.launch(share=PUBLIC_LINK,debug=False,show_error=True)

## 8. Summarise human outcomes, page stability, and verdict

In [ ]:
review_path=HUMAN/'human_alignment_reviews.csv'; human=pd.read_csv(review_path,dtype=str).fillna('') if review_path.exists() else pd.DataFrame(columns=['candidate_id','human_semantic_label','grouping_judgement','correction_action'])
if len(human):
    ev=scores.merge(human,on='candidate_id',validate='many_to_one'); ev['human_match']=ev.semantic_decision==ev.human_semantic_label; ev.to_csv(TABLES/'human_representation_evaluation.csv',index=False); hs=ev.groupby('representation',as_index=False).agg(reviewed=('candidate_id','nunique'),human_matches=('human_match','sum'),human_accuracy=('human_match','mean'),coverage=('semantic_decision',lambda x:float((x!='UNCERTAIN').mean()))); hs.to_csv(TABLES/'human_representation_summary.csv',index=False); human.groupby('grouping_judgement',as_index=False).size().to_csv(TABLES/'human_grouping_judgements.csv',index=False); human.groupby('correction_action',as_index=False).size().to_csv(TABLES/'human_correction_actions.csv',index=False)
else:
    for name in ['human_representation_evaluation.csv','human_representation_summary.csv','human_grouping_judgements.csv','human_correction_actions.csv']: pd.DataFrame().to_csv(TABLES/name,index=False)
page_summary=scores.assign(accepted=scores.semantic_decision!='UNCERTAIN').groupby(['representation','page'],as_index=False).agg(components=('candidate_id','size'),accepted=('accepted','sum'),coverage=('accepted','mean'),mean_probability=('top_probability','mean'),mean_disagreement=('ensemble_disagreement','mean')); page_summary.to_csv(TABLES/'representation_results_by_page.csv',index=False); comparison[comparison.representation_disagreement].to_csv(TABLES/'representation_disagreements.csv',index=False); comparison[comparison.alignment_regression].to_csv(TABLES/'alignment_regressions.csv',index=False)
cov={r:float(summary.loc[summary.representation==r,'coverage'].iloc[0]) for r in summary.representation}; complete=len(human)==len(review_queue) and len(review_queue)>0; verdict='PASS_ALIGNMENT_AUDIT_COMPLETE' if complete else ('PASS_ENGINEERING_PARTIAL_HUMAN_REVIEW' if len(human) else 'PASS_ENGINEERING_HUMAN_REVIEW_PENDING')
record={'version':'Notebook68-component-semantic-alignment-audit','created_utc':datetime.now(timezone.utc).isoformat(),'verdict':verdict,'pilot09_run':str(PILOT),'59R6_run':str(R6),'bundle_id':freeze.get('bundle_id'),'policy_id':policy.get('policy_id'),'pages':len(pages),'components':len(components),'coverage':cov,'raw_to_merged_resolved':int(comparison.raw_to_merged_resolved.sum()),'raw_to_alignment_resolved':int(comparison.raw_to_alignment_resolved.sum()),'alignment_regressions':int(comparison.alignment_regression.sum()),'human_review_rows':len(human),'human_review_queue_rows':len(review_queue),'human_review_complete':complete,'automatic_correction_authorised':False,'production_promotion_authority':False}
atomic_json(OUT/'notebook68_verdict.json',record); print(json.dumps(record,indent=2))

## 9. Final upload package

In [ ]:
env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}; atomic_json(OUT/'software_environment.json',env)
lines=['NOTEBOOK 68 COMPONENT-TO-SEMANTIC-REGION ALIGNMENT AUDIT','',f'Verdict: {record["verdict"]}',f'Pages/components: {len(pages)} / {len(components)}',f'RAW coverage: {cov.get("RAW",0):.4f}',f'59R5 merged coverage: {cov.get("59R5_MERGED",0):.4f}',f'Alignment proposal coverage: {cov.get("ALIGNMENT_PROPOSAL",0):.4f}',f'RAW uncertainty resolved by merge: {record["raw_to_merged_resolved"]}',f'RAW uncertainty resolved by alignment: {record["raw_to_alignment_resolved"]}',f'Alignment regressions: {record["alignment_regressions"]}',f'Human reviews: {len(human)} / {len(review_queue)}','','No automatic correction or production promotion is authorised.']; (OUT/'executive_summary.txt').write_text('\n'.join(lines)); (UPLOAD/'README.txt').write_text('Start with executive_summary.txt and notebook68_verdict.json. Human evidence is separate from frozen inference.\n')
artefacts=[OUT/'executive_summary.txt',OUT/'notebook68_verdict.json',OUT/'software_environment.json',AUDIT/'59R6_bundle_resolution.json',AUDIT/'pilot09_page_input_audit.csv',AUDIT/'tensor_identity.csv',TABLES/'representation_manifest.csv',TABLES/'representation_summary.csv',TABLES/'matched_representation_comparison.csv',TABLES/'representation_results_by_page.csv',TABLES/'two_axis_matrix.csv',TABLES/'human_review_queue.csv',TABLES/'human_representation_summary.csv',TABLES/'human_grouping_judgements.csv',TABLES/'human_correction_actions.csv',TABLES/'representation_disagreements.csv',TABLES/'alignment_regressions.csv']
if review_path.exists(): artefacts.append(review_path)
manifest=[]
for src in artefacts:
    require(src.exists(),f'Missing package file {src}'); rel=src.relative_to(OUT); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
for src in sorted(CROPS.glob('*.png'))[:60]:
    rel=Path('review_crops')/src.name; dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(src,dst); manifest.append({'relative_path':str(rel),'bytes':dst.stat().st_size,'sha256':sha256_file(dst)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False); (UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in manifest)); zip_path=OUT/'notebook68_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file(): z.write(p,p.relative_to(UPLOAD))
print('\n'.join(lines)); print('Bundle:',zip_path)